# What actually couples the darts of a visit

Notebook 19 found the project's oldest assumption failing: the three darts of a visit are not
independent draws. Hitting the treble 20 raised the next dart's chance of doing the same by
18 points. This notebook asks the follow-up question, which is the one that changes the code:
**what should replace the assumption?**

The answer turns out to require unpicking that 18 points first, because it is not one effect.
Three separate things are mixed into it, and only the third is about the throw at all:

1. **The aim moves.** A player who misses the treble 20 often works *down* the board -- to
   the 19, then the 18, then the 17. After a miss, the next dart is frequently not *aimed* at
   the treble 20, so of course it hits it less often. This is a decision, and the model has no
   state for it: in this project the aim point depends on the score and the dart index, and
   never on where the previous dart landed.
2. **The tails are wrong.** A Gaussian tight enough to hit the treble 20 at a professional
   rate puts essentially nothing in the double 20 or off the board. Real players land there
   percents of the time.
3. **Whatever is left.** Once the aim and the tails are in the model, is there any coupling
   between darts remaining -- and if so, what shape?

The method is the same throughout: write the models down first, derive statistics that tell
them apart, check on simulated data that those statistics really do tell them apart, and only
then fit anything real -- on a training split, judged on held-out visits.

In [ ]:
import os
import sys
module_path = os.path.abspath(os.path.join('..', '..'))
if module_path not in sys.path:
    sys.path.append(module_path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25,
    'grid.linewidth': 0.6, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 11, 'font.size': 9, 'legend.frameon': False,
})

from darts.calibration import SCORING_FLOOR
from darts.dependence import (BedGrid, VisitModel, encode_visits, signatures,
                              treble_centre_mm, TARGETS)

DATA = os.path.join(module_path, 'data', 'real')
RES = os.path.join(module_path, 'results', 'dependence')
if not os.path.exists(os.path.join(DATA, 'per_dart.csv')):
    raise SystemExit('run scripts/build_real_data.py first (see data/real/README.md)')

GRID = BedGrid(512)
KEY = ['source', 'player', 'leg_id', 'visit_index']

per_dart = pd.read_csv(os.path.join(DATA, 'per_dart.csv'), low_memory=False)
d = per_dart[(per_dart.post_bust_visit == 0) & per_dart.dart_index.isin([1, 2, 3])]
info = d.groupby(KEY).agg(n=('dart_index', 'size'), start=('score_before', 'max'),
                          total=('value', 'sum')).reset_index()
# only visits that start high enough that the scoring filter cannot select on
# their outcome -- notebook 19 showed selection biases the dependence downward
ok = info[(info.n == 3) & ((info.start - info.total) >= SCORING_FLOOR)
          & (info.start >= SCORING_FLOOR + 180)]
scoring = d.merge(ok[KEY], on=KEY)
VIS = (scoring.pivot_table(index=KEY, columns='dart_index', values='bed',
                           aggfunc='first').dropna().reset_index())
print(f'{len(VIS):,} unfiltered pure-scoring visits, {VIS.player.nunique()} players')

## 1 · The aim moves

The tell is the treble 19. It sits eleven segments from the treble 20, on the opposite side of
the board -- no dart aimed at one can land in the other by accident. So a dart near the 19 is
direct evidence of where it was *aimed*, which is the one thing a scoresheet normally cannot
give us. The same argument extends to the 18 and the 17, and it turns out all four are needed.

A dart is assigned to a target when it lands within one segment of it, which leaves about 1%
of darts unassigned rather than forcing them somewhere.

In [ ]:
from darts.dependence import BOARD_ORDER

def nearest_target(bed):
    # which target a dart was aimed at, when it landed within one segment
    if bed in ('MISS', '25', 'BULL'):
        return None
    number = int(bed[1:])
    best, best_d = None, 99
    for t in TARGETS:
        d = abs((BOARD_ORDER.index(number) - BOARD_ORDER.index(t) + 10) % 20 - 10)
        if d < best_d:
            best, best_d = t, d
    return best if best_d <= 1 else None

for i in (1, 2, 3):
    VIS[f'aim{i}'] = VIS[i].map(nearest_target)

print('which target each dart was thrown at, by position in the visit:')
share = pd.DataFrame({f'dart {i}': VIS[f'aim{i}'].value_counts(normalize=True,
                                                               dropna=False)
                      for i in (1, 2, 3)}).fillna(0.0)
share.index = [('unassigned' if pd.isna(i) else f'the {int(i)}') for i in share.index]
print(share.round(3).to_string())

print('\nwhere the next dart goes, given the one before it:')
rows = []
for frm in TARGETS:
    for hit_it, label in [(True, 'hit the treble'), (False, 'missed')]:
        m = pd.concat([
            VIS[(VIS[f'aim{d}'] == frm)
                & (VIS[d].str.startswith('T') == hit_it)
                & VIS[f'aim{d + 1}'].notna()].rename(
                    columns={f'aim{d + 1}': 'next'})[['next']]
            for d in (1, 2)])
        if len(m) < 100:
            continue
        row = {'from': f'the {frm}', 'after': label, 'n': len(m)}
        row.update({f'-> {int(t)}': (m['next'] == t).mean() for t in TARGETS})
        rows.append(row)
pd.DataFrame(rows).set_index(['from', 'after']).round(3)

The aim is not fixed, and it is not even two targets. Professionals use **four** and work
down them.

Dart 1 is at the 20 almost always -- 96.6%, with 3.2% landing too far from any target to
assign. By dart 3 the 20 holds only 66.7%, with 24.2% at the 19, 5.3% at the 18 and 2.0% at
the 17.

The transitions say why, and they are almost all **one step down**. From the 20, a dart that
hits the treble is followed by another at the 20 95.1% of the time; a dart that misses is
followed by one at the 19 24.8% of the time. From the 19 the same pattern repeats one place
along: stay 92.6% after a hit, move to the 18 35.7% after a miss.

So the rule is simple enough to write in one line -- *stay if you hit, else you may step one
place down* -- and it needs exactly two numbers. What it does not fit into is this project's
solver, where the aim point is a function of the score and the dart index and nothing else.
There is no state for "where did my last dart land".

In [ ]:
def lift(a, b):
    h, m = b[a == 1].mean(), b[a == 0].mean()
    n1, n0 = int((a == 1).sum()), int((a == 0).sum())
    se = np.sqrt(h * (1 - h) / n1 + m * (1 - m) / n0)
    return 100 * (h - m), 100 * se

t20 = {i: (VIS[i] == 'T20').astype(float) for i in (1, 2, 3)}
treb = {i: VIS[i].str.startswith('T').astype(float) for i in (1, 2, 3)}
rows = [('hit T20 (what notebook 19 measured)', *lift(t20[1], t20[2])),
        ('hit ANY treble (target-invariant)', *lift(treb[1], treb[2]))]
aims = VIS[['aim1', 'aim2', 'aim3']]
pure = VIS[(aims.isin([20]) | aims.isna()).all(axis=1)]
rows.append(('hit T20, visits that never move off the 20',
             *lift((pure[1] == 'T20').astype(float), (pure[2] == 'T20').astype(float))))
out = pd.DataFrame(rows, columns=['dart 1 -> dart 2', 'lift (pts)', 'se'])
out['z'] = out['lift (pts)'] / out.se
out.round(2)

About half of it is the aim, not the throw.

Notebook 19's statistic scores a dart by whether it hit the **treble 20** -- which measures the
throw only if the dart was aimed there. On these visits it reads **+22.33**. Score instead by
whether the dart hit **any** treble, a statistic that does not care which target it was thrown
at, and the same darts give **+13.11**. Restrict to the visits where the player never leaves
the 20, and it is **+10.95**.

All three are overwhelming -- the smallest is z = 14 -- so a real coupling survives. But the
headline number is roughly **twice** the part of it that is about throwing, and the difference
is a decision the player made, not a property of their throw.

Notebook 19's three robustness checks did not miss this through weakness. Pooling, drift and
selection were each ruled out properly. Switching target is a within-visit, within-player
effect that vanishes across the gap between visits, so it passes all three -- it is the
alternative that was not thought of, which is a different kind of gap from one that was tested
and survived.

Before building anything on that, it is worth checking it is not an artefact of one feed's
coding. Two things to confirm: that the bed labels agree with the scores recorded alongside
them, and that the pattern appears in *both* independent sources -- different providers,
different years, different tournaments.

In [ ]:
def bed_value(name):
    if name == 'MISS':
        return 0
    if name == '25':
        return 25
    if name == 'BULL':
        return 50
    return int(name[1:]) * {'S': 1, 'D': 2, 'T': 3}[name[0]]

check = scoring.dropna(subset=['bed']).copy()
check['implied'] = check.bed.map(bed_value)
mismatch = int((check.implied != check.value).sum())
print(f'darts whose bed disagrees with the score recorded beside it: '
      f'{mismatch} of {len(check):,} ({100 * mismatch / len(check):.3f}%)')

rows = []
for src, g in VIS.groupby('source'):
    hit = g[g[1] == 'T20']
    miss = g[(g.aim1 == 20) & (g[1] != 'T20')]
    if len(miss) < 50:
        continue
    rows.append({'source': src, 'visits': len(g),
                 'dart 1 at the 19': (g.aim1 == 19).mean(),
                 'dart 3 below the 20': (g.aim3.isin([19, 18, 17])).mean(),
                 'stays after a hit': (hit.aim2 == 20).mean(),
                 'moves after a miss': (miss.aim2 != 20).mean()})
pd.DataFrame(rows).set_index('source').round(3)

It is not a coding artefact.

The bed labels agree with the scores recorded beside them for all but a handful of darts in
three hundred thousand -- the build script gates on exactly this, so a feed that disagreed with
itself would have stopped the rebuild rather than reached here.

More importantly, the pattern appears in **both** sources: different commercial providers,
different years, different tournaments. The share of first darts at the 19, the share of third
darts below the 20, the probability of staying after a hit and of moving after a miss all agree
closely between them. Two feeds five years apart do not invent the same behaviour.

### Does the switch cost them anything?

Notebook 18 established that above a remaining score of 250 the treble 20 is the optimal aim
for every ability the model covers -- that is what makes the whole scoring-phase analysis
possible. So the model says this switch is a mistake. The data can price it.

The comparison has to be made **within a player**, since players differ both in how often they
switch and in how well they throw. And it is observational, not causal: a player decides to
switch knowing things about the dart that just landed which the bed label does not record, so
what follows is what switching is *associated* with, not what it *causes*.

In [ ]:
from darts.dependence import bed_geometry

# Every bed gets a value and a target, so that nothing is dropped. Excluding the
# beds that sit near neither target would be selecting on dart 2's *outcome* --
# and the beds it would exclude are the worst darts, which is exactly the
# comparison being made. Instead each dart is assigned to whichever target it is
# angularly nearer, the same rule the signature statistics use.
angle, _ = bed_geometry(GRID)
stack = np.stack([angle[t] for t in TARGETS])
nearest = np.argmin(np.where(np.isnan(stack), np.inf, np.abs(stack)), axis=0)
aimed_at = {n: TARGETS[nearest[i]] for i, n in enumerate(GRID.names)}

missed = VIS[(VIS.aim1 == 20) & (VIS[1] != 'T20')].copy()
missed['score2'] = missed[2].map(bed_value)
missed['moved'] = missed[2].map(aimed_at) != 20     # every dart gets a target
assert missed.score2.notna().all()

rows = []
for (src, pl), g in missed.groupby(['source', 'player']):
    stay, move = g[~g.moved], g[g.moved]
    if min(len(stay), len(move)) < 40:
        continue
    diff = move.score2.mean() - stay.score2.mean()
    se = np.sqrt(move.score2.var() / len(move) + stay.score2.var() / len(stay))
    rows.append({'player': pl, 'n stay': len(stay), 'n move': len(move),
                 'stayed at 20': stay.score2.mean(), 'moved down': move.score2.mean(),
                 'diff': diff, 'se': se})
cost = pd.DataFrame(rows).sort_values('diff')
w = 1 / cost.se ** 2
print(f'{len(cost)} players with at least 40 of each')
print(f'pooled difference in dart-2 score, moving vs staying: '
      f'{(w * cost["diff"]).sum() / w.sum():+.2f} +/- {np.sqrt(1 / w.sum()):.2f} points')
print(f'players where moving scored less: {(cost["diff"] < 0).sum()} of {len(cost)}')
cost.round(2).head(10)

In [ ]:
# what the model says the switch should cost, for comparison
vals = np.array([bed_value(n) for n in GRID.names], float)
rows = []
for sigma in (5.0, 6.5, 8.0, 10.0, 13.0, 16.0):
    p = {t: 0.87 * GRID.bed_pmf(t, np.zeros(2), sigma)
            + 0.13 * GRID.wide_pmf(t, sigma * 6) for t in TARGETS}
    rows.append({'sigma (mm)': sigma,
                 'E[score] at T20': p[20] @ vals, 'E[score] at T19': p[19] @ vals,
                 'cost of switching': (p[19] @ vals) - (p[20] @ vals),
                 'P(T20)': p[20][GRID.names.index('T20')],
                 'P(T19)': p[19][GRID.names.index('T19')]})
print('the 20 is flanked by the 1 and the 5 (mean 3); the 19 by the 3 and the 7 (mean 5)')
pd.DataFrame(rows).set_index('sigma (mm)').round(2)

Almost nothing, and the board's numbering is why.

Pooled within player, moving down after a miss is worth **-0.45 +/- 0.47 points** on the next
dart -- indistinguishable from zero, with 16 of 29 players scoring less and 13 more. The model
expects it to cost 1.2 to 1.5 points at a professional's spread, so the observed cost is if
anything *smaller* than predicted, by about one and a half standard errors.

The reason the cost is so small is in the second table. A treble 19 is worth three points less
than a treble 20, but the 20 is flanked by the **1 and the 5** and the 19 by the **3 and the
7**. Missing the 19 sideways pays a mean of 5 where missing the 20 pays 3, and that nearly
cancels the treble being smaller. At a 16 mm spread it cancels exactly.

This is a habit the model calls a mistake, performed by the best players in the world, that
costs them close to nothing. It is also a hint about *why* they do it, since the model can see
no reason at all: what changes between dart 1 and dart 2 is that there is now a dart in the
board, and nothing in this project represents that.

## 2 · The tails are wrong

The second thing the data forces into the model has nothing to do with dependence. Take the
player with the most clean scoring visits and look at where his *first* dart goes -- the one
dart we can be sure was aimed at the treble 20.

In [ ]:
who = VIS.groupby(['source', 'player']).size().idxmax()
sub = VIS[(VIS.source == who[0]) & (VIS.player == who[1])]
beds, hit = encode_visits(sub[[1, 2, 3]].values, GRID)
obs = np.bincount(beds[:, 0], minlength=GRID.n_beds) / len(beds)
top = np.argsort(obs)[::-1][:8]

rows = {'OBSERVED (dart 1)': obs[top]}
for sigma in (6.5, 8.0, 11.0, 14.5):
    rows[f'gaussian {sigma} mm'] = GRID.bed_pmf(20, np.zeros(2), sigma)[top]
tail = pd.DataFrame(rows, index=[GRID.names[i] for i in top]).T
print(f'{who[1]}: {len(beds):,} visits')
tail.round(4)

No Gaussian can do this.

Look at the last three columns. This player puts **1.8% of his first darts in the double 20**
and **1.7% off the board entirely** -- and a Gaussian tight enough to hit the treble 20 at his
observed rate puts, to four decimal places, **nothing** in either. The double 20 is 59 mm from
the treble's centre; at the 7 mm spread his treble rate implies, that is more than eight
standard deviations.

Widening the throw does not rescue it. It reaches the far beds only by abandoning the near
ones: by 14.5 mm the model finally produces some spread, and by then it predicts the treble 20
at 16% against an observed 36%. There is no single width that produces a sharp peak *and* a
heavy tail, because a Gaussian has only one parameter and this distribution needs two.

Fitted without a wide component, the optimiser splits the difference and lands on 14.5 mm --
which is why the naive fit reports professionals throwing twice as loosely as they do. The
cheapest repair is a contaminating component: most darts from the tight throw, a small
fraction from something much broader.

In [ ]:
core = GRID.bed_pmf(20, np.zeros(2), 6.9)
wide = GRID.wide_pmf(20, 6.9 * 6.0)
mix = 0.87 * core + 0.13 * wide
comp = pd.DataFrame({'OBSERVED': obs[top], 'gaussian 6.9 mm': core[top],
                     '87% core + 13% wide': mix[top]},
                    index=[GRID.names[i] for i in top]).T
comp.round(4)

## 3 · The model family

Six models, each adding one mechanism to the last, so that every comparison isolates one
thing. All of them keep the throw isotropic: notebook 12 showed shape matters, but letting it
vary here would confound the question being asked.

| | mechanism added | what it says |
|---|---|---|
| **A** | -- | one target, Gaussian, independent darts. **What the project assumes today** |
| **B** | the aim rule | after a miss the player may move to the treble 19 |
| **C** | a wide component | a fraction `eps` of darts come from a much wider throw |
| **D** | a shared **offset** | the visit's aim point is drawn once: darts cluster in *direction* |
| **E** | a shared **scale** | the visit is tight or loose: darts agree in *magnitude* only |

A sixth model with both couplings at once was fitted and dropped: on the few hundred visits one
player supplies, eight parameters are not identified -- the optimiser puts half the darts in
the wide component, collapses `sigma` to 2.7mm, and scores *worse* on held-out visits than
either coupling alone. That is a fact about the sample size, not about the mechanism.

D and E are the two simple ways to break independence, and they are not the same story. D is
"he had that stance for those three darts". E is "he was in the groove that visit". They make
different, checkable predictions, which is the next section.

Everything is fitted to *bed sequences* by exact maximum likelihood. The per-visit latent is
integrated out by Gauss-Hermite quadrature; the latent target of each dart is summed out
exactly with a two-state forward pass, which is cheap because a visit is three darts long.

### The statistics that tell D from E

Pre-specified, before any fitting, and computed by one function applied identically to real
visits and to simulated ones.

* **`dir_corr`** -- the correlation between the *signed* angular offsets of two darts in a
  visit, measured in segments from whichever target they were thrown at. A shared offset
  pushes a whole visit one way round the board, so this is positive. A shared scale has no
  preferred direction, so this is zero.
* **`mag_corr`** -- the same for *absolute* offsets. Both models make this positive.

So the ratio separates them: D couples direction and magnitude about equally, E couples
magnitude only. Before trusting that, it has to be shown on data where the truth is known.

In [ ]:
rng = np.random.default_rng(0)
spec = {'M0 independent': dict(switching=True),
        'M1 shared offset': dict(shared_offset=True, switching=True),
        'M2 shared scale': dict(shared_scale=True, switching=True)}
truth = {'M0 independent': [np.log(8.6), 0.0, -3.5, -1.0],
         'M1 shared offset': [np.log(7.0), 0.0, np.log(5.0), -3.5, -1.0],
         'M2 shared scale': [np.log(8.35), 0.0, np.log(0.42), -3.5, -1.0]}

rows = []
for name, kw in spec.items():
    m = VisitModel(GRID, **kw)
    b, h = m.simulate(np.array(truth[name]), 12000, rng=rng)
    s = signatures(b, h, GRID)
    rows.append({'simulated from': name, 'P(T20)': s['p_t20'],
                 'T20 lift 1->2': s['t20_lift_12'],
                 'treble lift 1->2': s['treble_lift_12'],
                 'dir_corr': s['dir_corr'], 'mag_corr': s['mag_corr']})
pd.DataFrame(rows).set_index('simulated from').round(3)

The two statistics do what they were designed to do.

Simulated from a **shared offset**, `dir_corr` is **+0.049** and `mag_corr` **+0.078** -- the
darts of a visit agree about direction and about distance roughly equally, because they share a
displacement. Simulated from a **shared scale**, `dir_corr` is **-0.006** and `mag_corr`
**+0.123**: the darts agree about how far out they are and not at all about which way, because
nothing has moved the aim. Simulated from **independent** darts, both are zero.

So the *ratio* separates the two mechanisms cleanly even though either one alone raises the
treble-20 lift. That is what makes the check on real data in section 5 worth running.

Note also the first row. Independent darts, with nothing coupling them, still produce a
treble-20 lift of **+7.9** -- purely from the aim rule. Any measurement of within-visit
dependence that does not model where the dart was aimed is measuring this as well.

## 4 · The fits

`scripts/dependence_fits.py` fits all six models to every player with at least 400 clean
scoring visits. The split is on whole **legs**, not visits -- visits inside a leg share a
player, an evening and a scoreline, so splitting on visits would leak information across the
boundary. Both halves contain the same players by design: the question is whether a model
generalises to new visits, not to new people.

Models are scored by held-out log-likelihood per visit, which needs no penalty term because
the parameters were never fitted to the visits doing the scoring.

In [ ]:
# What model A -- the project's current assumption -- makes of real darts.
# With one fixed target it has no way to produce a dart at the 19 at all, so the
# question is not how well it fits but how much of the data it calls impossible.
impossible = np.mean([(VIS[f'aim{i}'].isin([19, 18, 17])).mean() for i in (2, 3)])
p20 = GRID.bed_pmf(20, np.zeros(2), 13.1)
print('darts 2 and 3 thrown below the 20: %.1f%% of them' % (100 * impossible))
print('probability model A gives a dart at the S19: %.3g' % p20[GRID.names.index('S19')])
print('\nso model A is not merely a worse fit -- it assigns essentially zero')
print('probability to a fifth of the darts professionals actually throw.')

In [ ]:
fits = pd.read_csv(os.path.join(RES, 'fits.csv'))
sigs = pd.read_csv(os.path.join(RES, 'signatures.csv'))
print(f'{fits.player.nunique()} players, {fits.model.nunique()} models, '
      f'{fits.n_train.sum():,} training and {fits.n_test.sum():,} held-out visits')

# Model A is quoted separately below. Its log-likelihood is not a measure of
# fit but of impossibility: with no aim rule it gives every dart at the 19 a
# probability of zero, so its value is set by the numerical floor rather than by
# anything about the player. The ladder is therefore baselined on B.
base = (fits[fits.model == 'B + aim rule']
        .set_index(['source', 'player']).test_ll_per_visit)
fits['gain'] = fits.apply(
    lambda r: r.test_ll_per_visit - base.loc[(r.source, r.player)], axis=1)
ladder = fits.groupby('model').agg(
    players=('player', 'size'),
    gain=('gain', 'mean'),
    worst=('gain', 'min'),
    best=('gain', 'max'),
    sigma=('sigma', 'median'), tau=('tau', 'median'), nu=('nu', 'median'),
    eps=('eps', 'median'), s_miss=('s_miss', 'median')).round(4)
ladder

Three mechanisms, three very different sizes.

**The aim rule is the big one, and its number should not be read as a likelihood ratio.**
Model A cannot produce a dart at the 19 at all, so what it reports is not a poor fit but an
impossible one -- its value is set by the numerical floor, and the honest statement is the
qualitative one above: the project's current model gives essentially zero probability to a
quarter of the darts professionals throw after the first one of a visit. The ladder is therefore
quoted from B.

**The tails are the second, and they are worth more than everything else put together.**
Adding a wide component to a single dart buys **5.13 log-likelihood units per visit** and
improves every one of the 19 players. It also rescues `sigma`: model B, with no way to explain
a dart in the double 20, inflates the spread to a median of **13.8 mm**; model C recovers
**6.7 mm**, which is where professionals should be. A naive fit that ignores the tails reports
the best players in the world throwing twice as loosely as they do.

**The dependence is the third, and on this measure it is small.** A per-visit location offset
adds **0.008** a visit and helps 16 of 19 players; a per-visit *scale* adds **0.058** and helps
18 of 19, winning outright on 16. Set against 5.13 for the tails, both are marginal here --
whether this is the right measure of them is section 6.

In [ ]:
step = {'C + wide tail': 'B + aim rule',
        'D + shared offset': 'C + wide tail', 'E + shared scale': 'C + wide tail'}
piv = fits.pivot_table(index=['source', 'player'], columns='model',
                       values='test_ll_per_visit')
rows = []
for model, prev in step.items():
    delta = piv[model] - piv[prev]
    rows.append({'step': f'{prev.split()[0]} -> {model.split()[0]}',
                 'adds': model.split(' ', 1)[1],
                 'mean gain': delta.mean(), 'median': delta.median(),
                 'players improved': f'{(delta > 0).sum()} / {len(delta)}'})
pd.DataFrame(rows).round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 4.0))
order = ['B + aim rule', 'C + wide tail',
         'D + shared offset', 'E + shared scale']
for (src, pl), g in fits.groupby(['source', 'player']):
    g = g.set_index('model').reindex(order)
    ax.plot(range(len(order)), g.gain.values, marker='o', ms=3, lw=0.8,
            alpha=0.45, color='#2b6cb0')
mean = fits.groupby('model').gain.mean().reindex(order)
ax.plot(range(len(order)), mean.values, marker='o', ms=7, lw=2.4,
        color='#dd6b20', label='mean over players', zorder=5)
ax.set_xticks(range(len(order)))
ax.set_xticklabels([o.replace(' + ', '\n+ ') for o in order], fontsize=8)
ax.set_ylabel('held-out log-likelihood per visit,\nabove the aim-rule model')
ax.set_title('Each step adds one mechanism')
ax.legend()
fig.tight_layout()

## 5 · Does the winning model reproduce what the data does?

A better likelihood is not the same as a model that behaves like a darts player. The check is
the pre-specified signatures: draw visits from each fitted model and ask whether they show
what the real visits show. Nothing here was fitted to these statistics.

In [ ]:
cols = ['p_t20', 't20_lift_12', 'treble_lift_12', 'dir_corr', 'mag_corr', 'switch_rate']
w = sigs.groupby(['model'])[cols].mean()
n_obs = sigs[sigs.model == 'OBSERVED'].set_index(['source', 'player']).n
tab = w.reindex(['OBSERVED'] + order)
tab.round(3)

Here the small numbers turn out to matter.

Held-out likelihood said the coupling was worth 0.06 a visit. But the statistic notebook 19
actually reported -- the treble-20 lift from dart 1 to dart 2 -- is **25.1 points** in these
players' held-out visits, and the models reproduce it in a clean ladder: **0.1** with no aim
rule, **4.2** with the aim rule, **9.2** once the tails go in, **14.3** with a shared offset,
and **19.3** with a shared scale.

So the mechanisms are not competing to explain the same thing. The aim rule and the tails buy
almost all of the likelihood, because they govern where single darts go, which is most of what
a bed sequence tells you. The coupling buys almost none of it and yet is most of what
reproduces the *correlation* -- a bed is a coarse observation and a visit is only three darts
long, so a per-visit effect leaves little fingerprint on the likelihood while changing the
correlation completely. **Choosing a model by held-out likelihood alone would have understated
the one mechanism that matters for the quantity the MDP consumes.**

Two things this check does *not* support, and both are worth stating because the numbers look
like they might.

The **direction coupling is not evidence of anything**. Observed it is 0.015, against 0.002 to
0.008 from the models -- but its standard error across players is 0.011, because players differ
in it far more than any one player's visits do. It is 1.3 standard errors from zero and cannot
distinguish the models.

The **magnitude coupling is where the winning model visibly fails**. There the observed value
is 0.021 +/- 0.009 and model E produces 0.067 -- five standard errors too much. Model D, which
loses on every other measure, gets this one nearly right at 0.015. So E reproduces the headline
lift by over-cooking the mechanism underneath it, and neither model has the shape right.

In [ ]:
obs_row = sigs[sigs.model == 'OBSERVED'].set_index(['source', 'player'])
fig, axes = plt.subplots(1, 3, figsize=(10.4, 3.4))
for ax, stat, title in zip(axes,
                           ['t20_lift_12', 'dir_corr', 'mag_corr'],
                           ['T20 lift, dart 1 -> 2 (pts)',
                            'direction coupling', 'magnitude coupling']):
    for model, colour in [('C + wide tail', '#a0aec0'), ('D + shared offset', '#dd6b20'),
                          ('E + shared scale', '#2b6cb0')]:
        m = sigs[sigs.model == model].set_index(['source', 'player'])
        common = obs_row.index.intersection(m.index)
        ax.scatter(obs_row.loc[common, stat], m.loc[common, stat], s=16,
                   alpha=0.75, color=colour, label=model.split(' ', 1)[1])
    lo = min(ax.get_xlim()[0], ax.get_ylim()[0])
    hi = max(ax.get_xlim()[1], ax.get_ylim()[1])
    ax.plot([lo, hi], [lo, hi], color='k', lw=0.8, ls=':')
    ax.set_xlabel('observed'); ax.set_ylabel('model'); ax.set_title(title)
axes[0].legend(fontsize=7)
fig.tight_layout()

In [ ]:
best = fits.loc[fits.groupby(['source', 'player']).test_ll_per_visit.idxmax()]
print('model chosen per player, by held-out likelihood:')
print(best.model.value_counts().to_string())

win = fits[fits.model == 'E + shared scale'].copy()
# kappa is the wide component's width as a multiple of the core. Once eps is
# small it stops being identified from above: past roughly the radius of the
# board the wide component is just "somewhere on the board", and every larger
# value fits identically. Report the width in millimetres and say which players
# ran off the end rather than printing a number like 1e134.
win['wide sd (mm)'] = win.sigma * win.kappa
unbounded = win['wide sd (mm)'] > 225.5
print(f'\nwide component wider than the board, so its width is not identified: '
      f'{int(unbounded.sum())} of {len(win)} players')
win.loc[unbounded, 'wide sd (mm)'] = np.inf
show = win[['player', 'n_train', 'sigma', 'nu', 'eps', 'wide sd (mm)',
            's_hit', 's_miss']]
print(show.sort_values('n_train', ascending=False).round(3).to_string(index=False))

The winning model, per player, and what it says about them.

The median professional here throws with `sigma` **7.6 mm**, puts about **8.5%** of darts in a
wide component, moves off the 20 after **30%** of misses and **3%** of hits, and has a
per-visit scale of `nu` **0.35** -- their spread wandering by roughly a third either way from
one visit to the next. That is not a small description of a player, however small its
contribution to the likelihood was.

They also **pull sideways**, by a median of 1.3 mm and as much as 5 mm, and mostly the same
way: 14 of 19 pull towards the 5, which is the left side of the board as you face it. The fit
is not inventing this -- the per-player estimates correlate at **0.87** with the raw asymmetry
between the beds either side of the 20, and the board's segment boundaries were checked to be
exactly symmetric about the 20 before this was believed. Notebook 13 found a pull the most
expensive thing to get wrong, and notebook 18 found the sideways component the only one a
scoresheet measures well; this is that measurement, on real players.

Two honest caveats sit in this table. The width of the wide component is **not identified from
above**: once `eps` is small, anything broader than the board fits identically, and for several
players the fit runs off the end. And the throw is held isotropic throughout -- notebook 12
showed that shape matters, but letting it vary here would have confounded the question the
notebook is asking, so it is a stated simplification rather than a claim.

## 6 · Does the small gain matter for the game?

Held-out log-likelihood on beds is the right way to *choose* between these models, and the
wrong way to decide whether a mechanism matters. A per-visit scale of `nu = 0.4` means a
player's spread wanders by about half again either way from visit to visit -- a large physical
effect -- and it buys very little likelihood, because a bed is a coarse thing to observe and
most of that variation is invisible in it.

The quantity the project actually cares about is the **visit total**: it is what the MDP's
transition matrix is built from, and its spread is what notebook 19 found the model getting
wrong. So the last check is not a likelihood at all. Take each fitted model, draw visits from
it, add up the scores, and compare against what the player really did.

In [ ]:
# the fitted models, rebuilt from the stored parameters -- pack() inverts the
# unpack() the fitter uses, so nothing is refitted here
MODEL_KW = {'C + wide tail': dict(switching=True, contamination=True),
            'E + shared scale': dict(switching=True, contamination=True,
                                     shared_scale=True)}

def totals_of(beds):
    value = np.array([bed_value(n) for n in GRID.names])
    return value[beds].sum(axis=1)

rows = []
for (src, pl), sub in VIS.groupby(['source', 'player']):
    got = fits[(fits.source == src) & (fits.player == pl)]
    if got.empty:
        continue
    obs_beds, _ = encode_visits(sub[[1, 2, 3]].values, GRID)
    obs = totals_of(obs_beds)
    rows.append({'player': pl, 'model': 'OBSERVED', 'n': len(obs),
                 'mean': obs.mean(), 'sd': obs.std(),
                 'P(180) %': 100 * (obs == 180).mean(),
                 'P(140+) %': 100 * (obs >= 140).mean(),
                 'P(=60) %': 100 * (obs == 60).mean()})
    for model, kw in MODEL_KW.items():
        r = got[got.model == model].iloc[0]
        m = VisitModel(GRID, n_quad=7, **kw)
        theta = m.pack({'sigma': r.sigma, 'bias': [r.bias_x, 0.0], 'eps': r.eps,
                        'kappa': r.kappa, 'tau': r.tau, 'nu': r.nu,
                        's_hit': r.s_hit, 's_miss': r.s_miss})
        sim_b, _ = m.simulate(theta, 6000, rng=np.random.default_rng(5))
        t = totals_of(sim_b)
        rows.append({'player': pl, 'model': model, 'n': 6000,
                     'mean': t.mean(), 'sd': t.std(),
                     'P(180) %': 100 * (t == 180).mean(),
                     'P(140+) %': 100 * (t >= 140).mean(),
                     'P(=60) %': 100 * (t == 60).mean()})
tot = pd.DataFrame(rows)
cols = ['mean', 'sd', 'P(180) %', 'P(140+) %', 'P(=60) %']
print(f'averaged over the {tot.player.nunique()} players fitted:')
tot.groupby('model')[cols].mean().reindex(['OBSERVED'] + list(MODEL_KW)).round(2)

This is the mechanism's real defence, and it is a strong one.

Both models get the **mean** right, as they must -- it is what `sigma` is fitted to. What
separates them is the **spread**. With independent darts the visit total has a standard
deviation of **35.9** against an observed **40.5**: too narrow, which is exactly the
over-dispersion notebook 19 found and could not explain. Add the per-visit scale and it is
**41.5** -- slightly wide now, but the error has gone from 11% short to 2% over.

The consequence shows up where it is most visible. The independent model predicts a maximum on
**4.68%** of visits against an observed **7.91%** -- **41% too few 180s**. The scale mixture
predicts **8.47%**, within 7% of the truth. P(140+) improves from 21.6 to 25.4 against an
observed 26.6.

So a mechanism worth **0.06 log-likelihood units per visit** -- a rounding error against the
5.13 the tails were worth -- is the difference between getting a professional's 180 rate wrong
by two fifths and getting it right. Held-out likelihood on beds is a fair way to *rank* these
models and a bad way to decide which of them matters, because it scores what a bed sequence
reveals rather than what the game depends on.

The overshoot from section 5 shows here too, in the one column that gets worse: P(exactly 60)
falls from 7.16 to 5.95 against an observed 7.96. The model buys its tails slightly too dearly
in the middle.

## Verdict

Notebook 19 measured something real and named it wrongly. Taken apart, the +22 points of
within-visit coupling is three mechanisms, and the smallest of them is the one this project
would have added.

**The aim moves, and that is the largest failure by far.** Professionals use four scoring
targets and step down them after a miss: from the 20, a miss goes to the 19 a quarter of the
time; from the 19 to the 18, better than a third. Dart 1 is at the 20 in 96.6% of visits and
dart 3 in 66.7%. On a target-invariant statistic the coupling drops from +22.3 to +13.1, so
about half of what 19 measured is this. It replicates across two independent feeds five years
apart, and it costs the players nothing measurable (**-0.45 +/- 0.47** points), because the 19
is flanked by the 3 and the 7 where the 20 is flanked by the 1 and the 5.

The important part is not the size but the kind. This is not a missing parameter, it is a
missing **state variable**: the solver's aim point is a function of the score and the dart
index, and there is nowhere to put "where did my last dart land". The current model gives
essentially zero probability to a quarter of the darts thrown after the first of a visit.

**A single dart's tails are far too thin, and this is worth more than everything else.**
A wide component on about 8.5% of darts buys **5.13 log-likelihood units per visit** against
0.06 for all the dependence work, improves all 19 players, and is the difference between
reporting a professional's spread as 13.8 mm and as 6.7 mm. Anyone fitting a throw from
scoresheets without it is measuring the tails and calling the answer the spread.

**What is left is a shared *scale*, not a shared offset.** The visit is tight or loose, rather
than thrown from a displaced stance: `nu` around 0.35, winning outright on 16 of 19 players
where the offset model wins on 3. On held-out likelihood it is worth almost nothing -- 0.06 a
visit against 5.13 for the tails -- and on the quantity the MDP is built from it is worth
almost everything. With independent darts a visit total is too narrow (standard deviation 35.9
against an observed 40.5) and a maximum comes up **41% too rarely**; with the per-visit scale
the spread is 41.5 and the 180 rate is within 7% of the truth. Both facts are true of the same
parameter, which is the strongest single lesson here: **a bed sequence and a leg of darts do
not reward the same model.**

**And the winning model is visibly the wrong shape.** It reproduces the treble-20 lift by
overshooting the magnitude coupling five standard errors -- 0.067 against an observed
0.021 +/- 0.009 -- while the offset model, which loses everywhere else, gets that one nearly
right. Neither reaches the observed lift of 25.1. The likeliest culprit is the aim rule still
being too crude: it reduces the previous dart to hit-or-miss, when a player surely responds to
*how* they missed, and a richer rule would take coupling away from the throw and give it back
to the decision. That is the next experiment and it needs no new data.

### What this does not say

Professionals only, scoring phase only, and only at scores high enough that no checkout is in
reach. The throw is isotropic throughout, by choice, so nothing here bears on notebook 12. The
width of the wide component is not identified from above. And the coupling terms are estimated
from a few hundred visits per player, which is why the model carrying both of them at once was
dropped rather than reported.

### One about method

Two of the three findings here came from checking a number that looked odd, not from a test.
The four-target aim rule surfaced because the third dart's far-miss rate was three times the
first's; the sideways pull surfaced because a fitted parameter came back as **exactly** zero
for twelve of nineteen players, which is what an optimiser returns when it has not moved a
coordinate at all. The test suite passed throughout both. Tests check that the machinery does
what it was told; neither of these was a fault in the machinery.